# People Flow Detection using YOLO, ByteTrack and Heatmap

This notebook:

- detects only people using a pretrained YOLO model;
- maintains unique IDs with ByteTrack;
- draws an upper **IN** line and lower **OUT** line;
- counts downward crossings of the upper line as **IN**;
- counts upward crossings of the lower line as **OUT**;
- draws bounding boxes, IDs and trajectories;
- saves an annotated output video;
- generates a final center-point presence heatmap.

**Colab recommendation:** Runtime → Change runtime type → **T4 GPU**.

In [ ]:
# Cell 1: Install the required package
!pip -q install -U ultralytics

In [ ]:
# Cell 2: Imports and project configuration
import os
import cv2
import torch
import shutil
import urllib.request
import subprocess
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from collections import defaultdict, deque
from tqdm.auto import tqdm
from ultralytics import YOLO
from IPython.display import Video, display, Image

VIDEO_URL = "https://media.roboflow.com/supervision/video-examples/people-walking.mp4"

WORK_DIR = Path("/content/people_flow_project")
WORK_DIR.mkdir(parents=True, exist_ok=True)

INPUT_VIDEO = str(WORK_DIR / "people-walking.mp4")
RAW_OUTPUT_VIDEO = str(WORK_DIR / "people_flow_raw.mp4")
OUTPUT_VIDEO = str(WORK_DIR / "people_flow_output.mp4")
HEATMAP_IMAGE = str(WORK_DIR / "final_heatmap.png")
HEATMAP_OVERLAY_IMAGE = str(WORK_DIR / "final_heatmap_overlay.png")
README_FILE = str(WORK_DIR / "README.md")

# Current Ultralytics pretrained nano model.
# Change to "yolo11n.pt" if you specifically need YOLO11.
MODEL_NAME = "yolo26n.pt"

CONFIDENCE_THRESHOLD = 0.30
IOU_THRESHOLD = 0.50
IMAGE_SIZE = 640

# Person is class 0 in the COCO dataset.
PERSON_CLASS_ID = 0

# Line positions are calculated as fractions of frame height.
# Adjust these after viewing the preview cell.
UPPER_LINE_RATIO = 0.38
LOWER_LINE_RATIO = 0.64

# Reject tiny one-frame vertical changes caused by box jitter.
MIN_VERTICAL_MOVEMENT = 2

# Radius used when accumulating each person's center in the heatmap.
HEAT_POINT_RADIUS = 16

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Device:", "GPU" if DEVICE == 0 else "CPU")
print("Working directory:", WORK_DIR)

In [ ]:
# Cell 3: Download the sample video and read its properties
if not os.path.exists(INPUT_VIDEO):
    print("Downloading video...")
    urllib.request.urlretrieve(VIDEO_URL, INPUT_VIDEO)

cap = cv2.VideoCapture(INPUT_VIDEO)
if not cap.isOpened():
    raise RuntimeError(f"Could not open input video: {INPUT_VIDEO}")

FRAME_WIDTH = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
FRAME_HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS = cap.get(cv2.CAP_PROP_FPS)
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if FPS <= 0:
    FPS = 30.0

UPPER_LINE_Y = int(FRAME_HEIGHT * UPPER_LINE_RATIO)
LOWER_LINE_Y = int(FRAME_HEIGHT * LOWER_LINE_RATIO)

cap.release()

print(f"Resolution: {FRAME_WIDTH} x {FRAME_HEIGHT}")
print(f"FPS: {FPS:.2f}")
print(f"Frames: {TOTAL_FRAMES}")
print(f"Upper IN line:  (0, {UPPER_LINE_Y}) to ({FRAME_WIDTH - 1}, {UPPER_LINE_Y})")
print(f"Lower OUT line: (0, {LOWER_LINE_Y}) to ({FRAME_WIDTH - 1}, {LOWER_LINE_Y})")

In [ ]:
# Cell 4: Preview the first frame and line positions
cap = cv2.VideoCapture(INPUT_VIDEO)
ok, preview_frame = cap.read()
cap.release()

if not ok:
    raise RuntimeError("Could not read the first video frame.")

preview = preview_frame.copy()

cv2.line(
    preview,
    (0, UPPER_LINE_Y),
    (FRAME_WIDTH - 1, UPPER_LINE_Y),
    (0, 255, 0),
    4,
)
cv2.line(
    preview,
    (0, LOWER_LINE_Y),
    (FRAME_WIDTH - 1, LOWER_LINE_Y),
    (0, 0, 255),
    4,
)

cv2.putText(
    preview,
    f"IN line: y={UPPER_LINE_Y}",
    (20, max(35, UPPER_LINE_Y - 12)),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.8,
    (0, 255, 0),
    2,
    cv2.LINE_AA,
)
cv2.putText(
    preview,
    f"OUT line: y={LOWER_LINE_Y}",
    (20, max(35, LOWER_LINE_Y - 12)),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.8,
    (0, 0, 255),
    2,
    cv2.LINE_AA,
)

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Line preview — modify UPPER_LINE_RATIO and LOWER_LINE_RATIO if needed")
plt.show()

print("For exact coordinates, take a screenshot of this frame and use:")
print("https://polygonzone.roboflow.com/")

In [ ]:
# Cell 5: Run person detection, ByteTrack, counting and heatmap accumulation

# Load the pretrained YOLO detector.
model = YOLO(MODEL_NAME)

cap = cv2.VideoCapture(INPUT_VIDEO)
if not cap.isOpened():
    raise RuntimeError(f"Could not open input video: {INPUT_VIDEO}")

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(
    RAW_OUTPUT_VIDEO,
    fourcc,
    FPS,
    (FRAME_WIDTH, FRAME_HEIGHT),
)
if not writer.isOpened():
    cap.release()
    raise RuntimeError("Could not create the output video writer.")

# Required tracking and counting state.
previous_centers = {}
track_history = defaultdict(lambda: deque(maxlen=30))
last_seen_frame = {}

counted_in_ids = set()
counted_out_ids = set()

in_count = 0
out_count = 0
frame_index = 0

# Presence intensity map. Each center point adds energy to this array.
heat_accumulator = np.zeros((FRAME_HEIGHT, FRAME_WIDTH), dtype=np.float32)

last_clean_frame = None

def color_for_id(track_id: int):
    # Deterministic color based on the tracker ID.
    hue = int((track_id * 37) % 180)
    hsv = np.uint8([[[hue, 220, 255]]])
    bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)[0, 0]
    return tuple(int(value) for value in bgr)

progress = tqdm(total=TOTAL_FRAMES if TOTAL_FRAMES > 0 else None)

while True:
    ok, frame = cap.read()
    if not ok:
        break

    frame_index += 1
    clean_frame = frame.copy()
    last_clean_frame = clean_frame

    # persist=True keeps track IDs consistent between consecutive frames.
    result = model.track(
        source=frame,
        persist=True,
        tracker="bytetrack.yaml",
        classes=[PERSON_CLASS_ID],
        conf=CONFIDENCE_THRESHOLD,
        iou=IOU_THRESHOLD,
        imgsz=IMAGE_SIZE,
        device=DEVICE,
        verbose=False,
    )[0]

    if result.boxes is not None and result.boxes.id is not None:
        boxes = result.boxes.xyxy.cpu().numpy()
        track_ids = result.boxes.id.int().cpu().tolist()
        confidences = result.boxes.conf.cpu().numpy()

        for box, track_id, confidence in zip(boxes, track_ids, confidences):
            x1, y1, x2, y2 = map(int, box)

            # Clamp coordinates to the video frame.
            x1 = max(0, min(x1, FRAME_WIDTH - 1))
            y1 = max(0, min(y1, FRAME_HEIGHT - 1))
            x2 = max(0, min(x2, FRAME_WIDTH - 1))
            y2 = max(0, min(y2, FRAME_HEIGHT - 1))

            # Bounding-box center, as required for counting and heatmap.
            center_x = int((x1 + x2) / 2)
            center_y = int((y1 + y2) / 2)
            current_center = (center_x, center_y)

            # Add this center to the accumulated presence heatmap.
            cv2.circle(
                heat_accumulator,
                current_center,
                HEAT_POINT_RADIUS,
                1.0,
                thickness=-1,
            )

            # Compare the current center with the same ID's previous center.
            if track_id in previous_centers:
                previous_x, previous_y = previous_centers[track_id]
                vertical_movement = center_y - previous_y

                crossed_upper_downward = (
                    previous_y < UPPER_LINE_Y <= center_y
                    and vertical_movement >= MIN_VERTICAL_MOVEMENT
                )

                crossed_lower_upward = (
                    previous_y > LOWER_LINE_Y >= center_y
                    and vertical_movement <= -MIN_VERTICAL_MOVEMENT
                )

                # IN: from above the upper line to below it, moving downward.
                if crossed_upper_downward and track_id not in counted_in_ids:
                    in_count += 1
                    counted_in_ids.add(track_id)

                # OUT: from below the lower line to above it, moving upward.
                if crossed_lower_upward and track_id not in counted_out_ids:
                    out_count += 1
                    counted_out_ids.add(track_id)

            previous_centers[track_id] = current_center
            last_seen_frame[track_id] = frame_index
            track_history[track_id].append(current_center)

            color = color_for_id(track_id)

            # Bounding box.
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            # Bounding-box center.
            cv2.circle(frame, current_center, 4, color, -1)

            # ID and confidence label.
            label = f"ID {track_id} | {confidence:.2f}"
            (text_width, text_height), baseline = cv2.getTextSize(
                label,
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                2,
            )
            label_y1 = max(0, y1 - text_height - baseline - 8)
            cv2.rectangle(
                frame,
                (x1, label_y1),
                (x1 + text_width + 8, y1),
                color,
                -1,
            )
            cv2.putText(
                frame,
                label,
                (x1 + 4, y1 - baseline - 4),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (255, 255, 255),
                2,
                cv2.LINE_AA,
            )

            # Recent trajectory.
            points = list(track_history[track_id])
            for point_index in range(1, len(points)):
                cv2.line(
                    frame,
                    points[point_index - 1],
                    points[point_index],
                    color,
                    2,
                    cv2.LINE_AA,
                )

    # Remove inactive center/history entries to keep memory bounded.
    if frame_index % 300 == 0:
        stale_after = int(FPS * 10)
        stale_ids = [
            track_id
            for track_id, last_frame in last_seen_frame.items()
            if frame_index - last_frame > stale_after
        ]
        for track_id in stale_ids:
            previous_centers.pop(track_id, None)
            track_history.pop(track_id, None)
            last_seen_frame.pop(track_id, None)

    # Draw the two required horizontal lines.
    cv2.line(
        frame,
        (0, UPPER_LINE_Y),
        (FRAME_WIDTH - 1, UPPER_LINE_Y),
        (0, 255, 0),
        4,
    )
    cv2.line(
        frame,
        (0, LOWER_LINE_Y),
        (FRAME_WIDTH - 1, LOWER_LINE_Y),
        (0, 0, 255),
        4,
    )

    cv2.putText(
        frame,
        "IN LINE - downward crossing",
        (15, max(30, UPPER_LINE_Y - 12)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2,
        cv2.LINE_AA,
    )
    cv2.putText(
        frame,
        "OUT LINE - upward crossing",
        (15, max(30, LOWER_LINE_Y - 12)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 0, 255),
        2,
        cv2.LINE_AA,
    )

    # Semi-transparent counter panel.
    panel = frame.copy()
    cv2.rectangle(panel, (15, 15), (305, 125), (0, 0, 0), -1)
    frame = cv2.addWeighted(panel, 0.60, frame, 0.40, 0)

    cv2.putText(
        frame,
        f"IN: {in_count}",
        (35, 62),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.15,
        (0, 255, 0),
        3,
        cv2.LINE_AA,
    )
    cv2.putText(
        frame,
        f"OUT: {out_count}",
        (35, 108),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.15,
        (0, 0, 255),
        3,
        cv2.LINE_AA,
    )

    writer.write(frame)
    progress.update(1)

progress.close()
cap.release()
writer.release()

if last_clean_frame is None:
    raise RuntimeError("No frames were processed.")

print("Processing complete.")
print("Final IN count:", in_count)
print("Final OUT count:", out_count)
print("Raw output:", RAW_OUTPUT_VIDEO)

In [ ]:
# Cell 6: Generate and save the final heatmap and overlay

# Smooth accumulated center-point presence.
heat_blurred = cv2.GaussianBlur(
    heat_accumulator,
    ksize=(0, 0),
    sigmaX=25,
    sigmaY=25,
)

if float(heat_blurred.max()) > 0:
    heat_normalized = cv2.normalize(
        heat_blurred,
        None,
        alpha=0,
        beta=255,
        norm_type=cv2.NORM_MINMAX,
    ).astype(np.uint8)
else:
    heat_normalized = np.zeros_like(heat_blurred, dtype=np.uint8)

heat_color = cv2.applyColorMap(heat_normalized, cv2.COLORMAP_JET)

# Keep low-activity regions close to the original frame.
heat_mask = heat_normalized > 8
blended = cv2.addWeighted(last_clean_frame, 0.45, heat_color, 0.55, 0)
heat_overlay = last_clean_frame.copy()
heat_overlay[heat_mask] = blended[heat_mask]

cv2.putText(
    heat_overlay,
    "Final People Presence Heatmap",
    (25, 45),
    cv2.FONT_HERSHEY_SIMPLEX,
    1.0,
    (255, 255, 255),
    3,
    cv2.LINE_AA,
)

cv2.imwrite(HEATMAP_IMAGE, heat_color)
cv2.imwrite(HEATMAP_OVERLAY_IMAGE, heat_overlay)

print("Saved:", HEATMAP_IMAGE)
print("Saved:", HEATMAP_OVERLAY_IMAGE)

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(heat_overlay, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Final people presence heatmap overlay")
plt.show()

In [ ]:
# Cell 7: Convert the raw video to browser-compatible H.264
ffmpeg_command = [
    "ffmpeg",
    "-y",
    "-loglevel",
    "error",
    "-i",
    RAW_OUTPUT_VIDEO,
    "-c:v",
    "libx264",
    "-pix_fmt",
    "yuv420p",
    "-movflags",
    "+faststart",
    OUTPUT_VIDEO,
]

try:
    subprocess.run(ffmpeg_command, check=True)
    print("H.264 video saved:", OUTPUT_VIDEO)
except (subprocess.CalledProcessError, FileNotFoundError):
    shutil.copyfile(RAW_OUTPUT_VIDEO, OUTPUT_VIDEO)
    print("FFmpeg conversion failed; copied the raw MP4 instead:", OUTPUT_VIDEO)

In [ ]:
# Cell 8: Create the README deliverable
readme_text = f"""
# People Flow Detection using YOLO and ByteTrack

## Detection method
- Pretrained model: `{MODEL_NAME}`
- Framework: Ultralytics YOLO
- Detected class: person only, COCO class ID `{PERSON_CLASS_ID}`
- Confidence threshold: `{CONFIDENCE_THRESHOLD}`
- IoU threshold: `{IOU_THRESHOLD}`

## Tracking method
- Tracker: ByteTrack through `tracker="bytetrack.yaml"`
- `persist=True` keeps object identities between consecutive frames.
- Each detection is displayed with its bounding box and unique tracker ID.

## Line coordinates
- Video resolution: `{FRAME_WIDTH} x {FRAME_HEIGHT}`
- Upper IN line: `(0, {UPPER_LINE_Y})` to `({FRAME_WIDTH - 1}, {UPPER_LINE_Y})`
- Lower OUT line: `(0, {LOWER_LINE_Y})` to `({FRAME_WIDTH - 1}, {LOWER_LINE_Y})`

## Counting logic
- For each tracker ID, the previous bounding-box center is stored.
- IN is counted when the center moves downward from above the upper line to the line or below it.
- OUT is counted when the center moves upward from below the lower line to the line or above it.
- Separate ID sets prevent repeated counting caused by small position jitter.

## Heatmap
- Every person's bounding-box center adds intensity to a floating-point accumulation map.
- The accumulated map is Gaussian blurred, normalized and converted to a JET color map.
- The notebook saves both a standalone heatmap and an overlay on the final clean video frame.

## Final results
- IN count: `{in_count}`
- OUT count: `{out_count}`

## Output files
- Annotated video: `{OUTPUT_VIDEO}`
- Standalone heatmap: `{HEATMAP_IMAGE}`
- Heatmap overlay: `{HEATMAP_OVERLAY_IMAGE}`
""".strip()

with open(README_FILE, "w", encoding="utf-8") as file:
    file.write(readme_text)

print(readme_text)
print("\nREADME saved:", README_FILE)

In [ ]:
# Cell 9: Display the output video and heatmap
display(Video(OUTPUT_VIDEO, embed=True, width=900))
display(Image(filename=HEATMAP_OVERLAY_IMAGE))

In [ ]:
# Cell 10: Download all deliverables from Colab
from google.colab import files

files.download(OUTPUT_VIDEO)
files.download(HEATMAP_IMAGE)
files.download(HEATMAP_OVERLAY_IMAGE)
files.download(README_FILE)